# Amazon ML Challenge 2026 - Metric Verification & Evaluation Notebook

## 📐 Understanding the Official Evaluation Metric: Macro $F_{0.5}$

The official competition score is calculated as:
$$
F_{0.5} = \frac{(1 + 0.5^2) \times \text{Precision} \times \text{Recall}}{0.5^2 \times \text{Precision} + \text{Recall}} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}
$$

### 💡 Why $F_{0.5}$ instead of standard $F_1$?
1. **Precision is weighted 4× as heavily as Recall**:
   $$\text{Weight of Precision} = \frac{1}{\beta^2} = \frac{1}{0.5^2} = 4$$
2. **The Business Intuition**:
   In e-commerce catalog entity resolution, **False Positives (incorrectly merging two distinct products)** are catastrophic—e.g. merging a *64GB iPhone 13* with a *128GB iPhone 14* causes incorrect pricing, misdirected warehouse shipments, customer complaints, and returns.
   In contrast, a **False Negative (failing to link two identical listings)** is minor—the catalog simply keeps two separate listings.

### 🎯 How Macro $F_{0.5}$ is Calculated Across Entities:
For each Source 1 product $e$:
- **Singleton ($e$ has no matches in S2/S3)**:
  - If predicted matches is empty: **Score = 1.0** (correctly identified singleton)
  - If predicted matches has any false link: **Score = 0.0** (false merge)
- **Matched Entity ($e$ has true matches in S2 and/or S3)**:
  - If predicted matches is empty: **Score = 0.0** (missed completely)
  - Otherwise: computes per-entity Precision and Recall, and calculates $F_{0.5}$.
- **Final Metric**: The arithmetic mean (Macro Average) over all entities.

---

In [ ]:
# 1. Environment Setup & Google Drive Mount (if running in Colab)
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
except Exception:
    print("Running locally or Drive already mounted.")

# Locate project root
project_dir = Path("/content/amazon-ml-challenge") if os.path.exists("/content/amazon-ml-challenge") else Path(".")
print(f"Project directory: {project_dir.resolve()}")

## 📂 2. Check the Status & Statistics of `output/gdrive2`

In [ ]:
# 2. Inspect output/gdrive2 metadata and files
gdrive2_dir = Path("output/gdrive2") if Path("output/gdrive2").exists() else Path("/content/drive/MyDrive/amazon-ml-challenge-2026/output")

print(f"Checking output directory: {gdrive2_dir}")
for f in sorted(gdrive2_dir.glob("*")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name:<30} {size_mb:8.2f} MB")

# Read completion tokens
for done_file in gdrive2_dir.glob("*.done"):
    print(f"\n[{done_file.name}]:")
    print(json.dumps(json.loads(done_file.read_text()), indent=2))

## 🔍 3. Submission Integrity Check: Validate `matching_results.tsv`

In [ ]:
# 3. Verify matching_results.tsv row count, structure, and singleton ratio
sub_path = gdrive2_dir / "matching_results.tsv"
if not sub_path.exists():
    sub_path = Path("output/runs/run_full_production_v1/matching_results.tsv")

print(f"Reading submission file: {sub_path}")

total_rows = 0
singletons = 0
matched = 0
preview = []

with open(sub_path, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split('\t')
    assert header == ["source_1_id", "matched_source_ids"], f"Invalid header: {header}"
    for idx, line in enumerate(f):
        total_rows += 1
        parts = line.strip().split('\t')
        s1_id = parts[0]
        matches = parts[1] if len(parts) > 1 and parts[1] else ""
        if not matches or matches == "[]":
            singletons += 1
        else:
            matched += 1
        if idx < 5:
            preview.append((s1_id, matches))

print(f"✅ Validation Successful:")
print(f"  * Total S1 Entities Evaluated: {total_rows:,} (Expected: 1,732,544)")
print(f"  * Matched Entities:           {matched:,} ({matched/total_rows*100:.2f}%)")
print(f"  * Singleton Entities:         {singletons:,} ({singletons/total_rows*100:.2f}%)")
print("\n--- First 5 Predictions ---")
for s1_id, match in preview:
    print(f"  {s1_id} -> {match}")

## 📊 4. Official Macro $F_{0.5}$ Evaluation Function

This function implements the official competition evaluation metric.

In [ ]:
# 4. Define Official Macro F_0.5 Evaluation
def evaluate_official_macro_f05(pred_dict, true_dict, all_s1_ids):
    """
    Calculates the Macro F_0.5 score across all Source 1 entities strictly adhering to competition rules.
    """
    singleton_scores = []
    matched_scores = []
    all_scores = []
    
    for s1_id in all_s1_ids:
        preds = set(pred_dict.get(s1_id, []))
        trues = set(true_dict.get(s1_id, []))
        
        # Case 1: Ground Truth is a Singleton (no duplicate listings in S2 or S3)
        if not trues:
            if not preds:
                score = 1.0  # Perfect: correctly identified as singleton
            else:
                score = 0.0  # False merge: wrongly linked a singleton
            singleton_scores.append(score)
            all_scores.append(score)
            continue
            
        # Case 2: Ground Truth has matches
        if not preds:
            score = 0.0      # Missed all matches
            matched_scores.append(score)
            all_scores.append(score)
            continue
            
        tp = len(preds & trues)
        fp = len(preds - trues)
        fn = len(trues - preds)
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        denom = 0.25 * p + r
        f05 = (1.25 * p * r) / denom if denom > 0 else 0.0
        
        matched_scores.append(f05)
        all_scores.append(f05)
        
    return {
        "macro_f05": float(np.mean(all_scores)),
        "singleton_f05": float(np.mean(singleton_scores)) if singleton_scores else 0.0,
        "matched_f05": float(np.mean(matched_scores)) if matched_scores else 0.0,
        "n_total": len(all_s1_ids),
        "n_singletons": len(singleton_scores),
        "n_matched": len(matched_scores)
    }

print("Evaluation function loaded.")

## 🧪 5. Benchmark the Trained Model on Ground Truth Validation Set

Here we load the trained GBDT model from `output/gdrive2/lgbm_entity_resolver.pkl` and evaluate its exact Macro $F_{0.5}$ on the validation ground truth.

In [ ]:
# 5. Evaluate the gdrive2 model on validation data
import pickle

model_path = gdrive2_dir / "lgbm_entity_resolver.pkl"
if not model_path.exists():
    model_path = Path("checkpoints/lgbm_entity_resolver.pkl")

if model_path.exists():
    print(f"Loading model: {model_path}")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    
    print("\n--- Model Architecture & Configuration ---")
    if hasattr(model, "best_threshold"):
        print(f"Optimal Decision Threshold:  {model.best_threshold:.4f}")
    if hasattr(model, "feature_names"):
        print(f"Number of Features:          {len(model.feature_names)}")
        print("Features:", model.feature_names)
    if hasattr(model, "classifier") and hasattr(model.classifier, "n_estimators"):
        print(f"Number of GBDT Trees:        {model.classifier.n_estimators}")
    
    # If scorecard was generated during training:
    sc_file = Path("output/scorecard.json")
    if sc_file.exists():
        sc = json.loads(sc_file.read_text())
        print("\n--- Official Scorecard Metrics (from Validation Set) ---")
        print(f"  * Official Macro F0.5:      {sc.get('official_macro_f05', 0):.4f}")
        print(f"  * Classifier Pairwise F1:   {sc.get('classifier_pair_f1', 0):.4f}")
        print(f"  * Classifier Pair Precision:{sc.get('classifier_pair_precision', 0)*100:.2f}%")
        print(f"  * Classifier Pair Recall:   {sc.get('classifier_pair_recall', 0)*100:.2f}%")
        print(f"  * Singleton Sub-Score:      {sc.get('singleton_macro_f05', 0):.4f}")
        print(f"  * Matched Entities Sub-Score:{sc.get('matched_macro_f05', 0):.4f}")
else:
    print(f"Model file not found at {model_path}")

## 📈 6. Threshold Sensitivity: Why Precision Tuning Maximizes $F_{0.5}$

Because $F_{0.5}$ penalizes False Positives 4× more than False Negatives, raising the threshold $\tau$ from 0.20 to 0.35–0.45 eliminates false merges and substantially boosts the final score.

In [ ]:
# 6. Mathematical demonstration of Threshold vs F0.5
print("Mathematical Demonstration of F0.5 vs F1 for different Precision/Recall balances:")
cases = [
    ("High Precision (P=0.95, R=0.80)", 0.95, 0.80),
    ("Balanced Model (P=0.85, R=0.85)", 0.85, 0.85),
    ("High Recall    (P=0.75, R=0.95)", 0.75, 0.95),
]

print(f"{'Scenario':<34} | {'Precision':<9} | {'Recall':<7} | {'F1':<7} | {'Official F0.5':<12}")
print("-" * 78)
for name, p, r in cases:
    f1 = 2 * p * r / (p + r)
    f05 = (1.25 * p * r) / (0.25 * p + r)
    print(f"{name:<34} | {p:<9.2f} | {r:<7.2f} | {f1:<7.4f} | {f05:<12.4f} 🏆" if f05 == max(c[1] for c in cases) else f"{name:<34} | {p:<9.2f} | {r:<7.2f} | {f1:<7.4f} | {f05:<12.4f}")

print("\nNotice: The High-Precision model achieves the highest Official F0.5 score!")